In [28]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    accuracy_score,
    recall_score
)

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"

In [ ]:
df = pd.read_csv('/content/Alzheimer_Local_CNN_Ready.csv')
df.head()

,ID,Age,Gender,Educ,MMSE,Target,eTIV,nWBV,Image_Path
0,OAS1_0001_MR1,-0.114079,0,1.0,1.052156,0.0,-1.945250,-0.748104,C:\NU\images\disc1\OAS1_0001_MR1\PROCESSED\MPR...
1,OAS1_0002_MR1,-2.278127,0,3.0,1.052156,0.0,-4.687976,2.502448,C:\NU\images\disc1\OAS1_0002_MR1\PROCESSED\MPR...
2,OAS1_0003_MR1,-0.227977,0,3.0,0.836511,1.0,-0.413779,-2.446153,C:\NU\images\disc1\OAS1_0003_MR1\PROCESSED\MPR...
3,OAS1_0010_MR1,-0.114079,1,4.0,1.159978,0.0,2.120109,-3.367951,C:\NU\images\disc1\OAS1_0010_MR1\PROCESSED\MPR...
4,OAS1_0011_MR1,-2.619819,0,2.0,1.159978,0.0,-2.265467,3.327215,C:\NU\images\disc1\OAS1_0011_MR1\PROCESSED\MPR...


In [ ]:

X = df.drop(columns=['ID', 'Target','Image_Path'])
y = df['Target']

df.head()

,ID,Age,Gender,Educ,MMSE,Target,eTIV,nWBV,Image_Path
0,OAS1_0001_MR1,-0.114079,0,1.0,1.052156,0.0,-1.945250,-0.748104,C:\NU\images\disc1\OAS1_0001_MR1\PROCESSED\MPR...
1,OAS1_0002_MR1,-2.278127,0,3.0,1.052156,0.0,-4.687976,2.502448,C:\NU\images\disc1\OAS1_0002_MR1\PROCESSED\MPR...
2,OAS1_0003_MR1,-0.227977,0,3.0,0.836511,1.0,-0.413779,-2.446153,C:\NU\images\disc1\OAS1_0003_MR1\PROCESSED\MPR...
3,OAS1_0010_MR1,-0.114079,1,4.0,1.159978,0.0,2.120109,-3.367951,C:\NU\images\disc1\OAS1_0010_MR1\PROCESSED\MPR...
4,OAS1_0011_MR1,-2.619819,0,2.0,1.159978,0.0,-2.265467,3.327215,C:\NU\images\disc1\OAS1_0011_MR1\PROCESSED\MPR...


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [ ]:
# ============================================================
# 2. LINEAR SVM MODEL
# ============================================================

In [ ]:


linear_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(
        kernel='linear',
        class_weight='balanced',
        probability=True,
        random_state=42
    ))
])

linear_svm.fit(X_train, y_train)

y_pred_linear = linear_svm.predict(X_test)
y_prob_linear = linear_svm.predict_proba(X_test)[:, 1]

linear_accuracy = accuracy_score(y_test, y_pred_linear)

print("Linear SVM Results\n")
print(f"Accuracy: {linear_accuracy:.4f}\n")
print(classification_report(
    y_test,
    y_pred_linear,
    target_names=['Healthy (0)', 'Early-Stage (1)']
))

Linear SVM Results

Accuracy: 0.7887

                 precision    recall  f1-score   support

    Healthy (0)       0.81      0.83      0.82        41
Early-Stage (1)       0.76      0.73      0.75        30

       accuracy                           0.79        71
      macro avg       0.78      0.78      0.78        71
   weighted avg       0.79      0.79      0.79        71



In [ ]:
# ============================================================
# SVM cofusion matrix
# ============================================================

In [45]:
cm_linear = confusion_matrix(y_test, y_pred_linear)

fig_linear_cm = px.imshow(
    cm_linear,
    text_auto=True,
    color_continuous_scale='Purples',
    labels=dict(
        x="Predicted Diagnosis",
        y="Actual Diagnosis",
        color="Patients"
    ),
    x=['Healthy (0)', 'Early-Stage (1)'],
    y=['Healthy (0)', 'Early-Stage (1)'],
    title="Linear SVM Confusion Matrix"
)

fig_linear_cm.update_layout(title_x=0.5, width=600, height=600)
fig_linear_cm.show()

In [ ]:
# ============================================================
#Linear SVM ROC Curve
# ============================================================

In [46]:
fpr_linear, tpr_linear, thresholds_linear = roc_curve(y_test, y_prob_linear)
roc_auc_linear = auc(fpr_linear, tpr_linear)

fig_linear_roc = px.area(
    x=fpr_linear,
    y=tpr_linear,
    title=f'ROC Curve - Linear SVM (AUC = {roc_auc_linear:.4f})',
    labels=dict(
        x='False Positive Rate (1 - Specificity)',
        y='True Positive Rate (Sensitivity/Recall)'
    ),
    width=700,
    height=600,
    color_discrete_sequence=['#9467bd']
)

fig_linear_roc.add_shape(
    type='line',
    line=dict(dash='dash', color='gray'),
    x0=0,
    x1=1,
    y0=0,
    y1=1
)

fig_linear_roc.update_layout(title_x=0.5)
fig_linear_roc.show()

In [ ]:
# ============================================================
# 4. NON-LINEAR SVM MODEL TRAINING
# ============================================================

In [31]:
rbf_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(
        kernel='rbf',
        class_weight='balanced',
        probability=True,
        random_state=42
    ))
])

rbf_svm.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('svm',
                 SVC(class_weight='balanced', probability=True,
                     random_state=42))])

In [32]:
# ============================================================
# 5. RBF SVM PREDICTIONS & TEXT EVALUATION
# ===========================================================

In [33]:
y_pred_rbf = rbf_svm.predict(X_test)
y_prob_rbf = rbf_svm.predict_proba(X_test)[:, 1]

rbf_accuracy = accuracy_score(y_test, y_pred_rbf)

print("Non-Linear SVM Results - RBF Kernel\n")
print(f"Accuracy: {rbf_accuracy:.4f}\n")
print(classification_report(
    y_test,
    y_pred_rbf,
    target_names=['Healthy (0)', 'Early-Stage (1)']
))

Non-Linear SVM Results - RBF Kernel

Accuracy: 0.8169

                 precision    recall  f1-score   support

    Healthy (0)       0.87      0.80      0.84        41
Early-Stage (1)       0.76      0.83      0.79        30

       accuracy                           0.82        71
      macro avg       0.81      0.82      0.81        71
   weighted avg       0.82      0.82      0.82        71



In [ ]:
# ============================================================
#  RBF SVM Supporting Graphs
# ===========================================================

In [34]:
cm_rbf = confusion_matrix(y_test, y_pred_rbf)

fig_rbf_cm = px.imshow(
    cm_rbf,
    text_auto=True,
    color_continuous_scale='Purples',
    labels=dict(
        x="Predicted Diagnosis",
        y="Actual Diagnosis",
        color="Patients"
    ),
    x=['Healthy (0)', 'Early-Stage (1)'],
    y=['Healthy (0)', 'Early-Stage (1)'],
    title="Non-Linear SVM Confusion Matrix - RBF Kernel"
)

fig_rbf_cm.update_layout(title_x=0.5, width=600, height=600)
fig_rbf_cm.show()

In [35]:
# ============================================================
#RBF SVM ROC Curve
# ===========================================================

In [36]:
fpr_rbf, tpr_rbf, thresholds_rbf = roc_curve(y_test, y_prob_rbf)
roc_auc_rbf = auc(fpr_rbf, tpr_rbf)

fig_rbf_roc = px.area(
    x=fpr_rbf,
    y=tpr_rbf,
    title=f'ROC Curve - RBF SVM (AUC = {roc_auc_rbf:.4f})',
    labels=dict(
        x='False Positive Rate (1 - Specificity)',
        y='True Positive Rate (Sensitivity/Recall)'
    ),
    width=700,
    height=600,
    color_discrete_sequence=['#9467bd']
)

fig_rbf_roc.add_shape(
    type='line',
    line=dict(dash='dash', color='gray'),
    x0=0,
    x1=1,
    y0=0,
    y1=1
)

fig_rbf_roc.update_layout(title_x=0.5)
fig_rbf_roc.show()

In [ ]:
# ============================================================
# 4.7 Accuracy and AUC Comparison
# ===========================================================

In [37]:
comparison_df = pd.DataFrame({
    'Model': ['Linear SVM', 'Non-Linear SVM (RBF)'],
    'Accuracy': [linear_accuracy, rbf_accuracy],
    'AUC': [roc_auc_linear, roc_auc_rbf]
})

comparison_df

,Model,Accuracy,AUC
0,Linear SVM,0.788732,0.889431
1,Non-Linear SVM (RBF),0.816901,0.909756


In [41]:
fig_combined_roc = go.Figure()

fig_combined_roc.add_trace(go.Scatter(
    x=fpr_linear,
    y=tpr_linear,
    mode='lines',
    name=f'Linear SVM (AUC = {roc_auc_linear:.4f})'
))

fig_combined_roc.add_trace(go.Scatter(
    x=fpr_rbf,
    y=tpr_rbf,
    mode='lines',
    name=f'RBF SVM (AUC = {roc_auc_rbf:.4f})'
))

fig_combined_roc.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random Classifier',
    line=dict(dash='dash', color='gray')
))

fig_combined_roc.update_layout(
    title='Combined ROC Curve: Linear SVM vs RBF SVM',
    xaxis_title='False Positive Rate (1 - Specificity)',
    yaxis_title='True Positive Rate (Sensitivity/Recall)',
    title_x=0.5,
    width=750,
    height=600
)

fig_combined_roc.show()

In [43]:
# ============================================================
# 8. FINAL TUNED SVM EVALUATION
# ============================================================

y_pred_best_svm = best_svm.predict(X_test)
y_prob_best_svm = best_svm.predict_proba(X_test)[:, 1]

best_svm_accuracy = accuracy_score(y_test, y_pred_best_svm)

print("Final Tuned SVM Results\n")
print(f"Accuracy: {best_svm_accuracy:.4f}\n")
print(classification_report(
    y_test,
    y_pred_best_svm,
    target_names=['Healthy (0)', 'Early-Stage (1)']
))

Final Tuned SVM Results

Accuracy: 0.7042

                 precision    recall  f1-score   support

    Healthy (0)       0.67      0.98      0.79        41
Early-Stage (1)       0.91      0.33      0.49        30

       accuracy                           0.70        71
      macro avg       0.79      0.65      0.64        71
   weighted avg       0.77      0.70      0.66        71

